<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/DS_PROJECT/ds_proj_meth/CRISP_DM_Project_Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект: Анализ выживаемости пассажиров Титаника  
**Автор:** Кондратьев Степан  
**Дата:** 2025 март  
**Цель:** Использование методов машинного обучения для прогнозирования выживаемости пассажиров Титаника на основе их характеристик, таких как пол, возраст, класс кают и других факторов.  

## 1. Понимание бизнеса (Business Understanding)

**Цель проекта:** Прогнозирование выживаемости пассажиров Титаника на основе их характеристик.  

**Постановка задачи:** Бинарная классификация пассажиров на выживших (1) и погибших (0).  

**Критерии успеха:**  
- **Метрики:**  
  - Точность (Accuracy)  
  - F1-мера (F1-Score)  
  - ROC-AUC  
- **Бизнес-результат:**  
  - Понимание ключевых факторов, влияющих на выживаемость, для улучшения безопасности пассажиров в будущем.  
  - Создание модели, способной предсказывать вероятность выживания с высокой точностью.  

## 2. Понимание данных (Data Understanding)

### Импорт библиотек

In [ ]:
# Импорт библиотек
import polars as pl  # Для работы с данными: чтение, обработка и анализ

### Загрудка данных

In [ ]:
# Настройка отображения всех столбцов в Polars
pl.Config.set_tbl_cols(-1)  # Показывать все столбцы

# Загрузка данных
train_url = "https://raw.githubusercontent.com/stefkong1982/netology.ru/refs/heads/Master/DS_PROJECT/ds_proj_meth/train.csv"
test_url = "https://raw.githubusercontent.com/stefkong1982/netology.ru/refs/heads/Master/DS_PROJECT/ds_proj_meth/test.csv"

train_df = pl.read_csv(train_url)
test_df = pl.read_csv(test_url)

In [ ]:
# # Вывод данных для первичного анализа
# print("Данные обучающей выборки:")
# train_df

Данные обучающей выборки:

Размер: (891, 12)

| PassengerId | Survived | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket              | Fare     | Cabin   | Embarked |
|-------------|----------|--------|------------------------------------------------|--------|-------|-------|-------|---------------------|----------|---------|----------|
| i64         | i64      | i64    | str                                            | str    | f64   | i64   | i64   | str                 | f64      | str     | str      |
| 1           | 0        | 3      | "Braund, Mr. Owen Harris"                     | "male" | 22.0  | 1     | 0     | "A/5 21171"         | 7.25     | "Unknown" | "S"      |
| 2           | 1        | 1      | "Cumings, Mrs. John Bradley (Flo)             | "female" | 38.0  | 1     | 0     | "PC 17599"          | 71.2833  | "C85"   | "C"      |
| 3           | 1        | 3      | "Heikkinen, Miss. Laina"                      | "female" | 26.0  | 0     | 0     | "STON/O2. 3101282"  | 7.925    | "Unknown" | "S"      |
| 4           | 1        | 1      | "Futrelle, Mrs. Jacques Heath (Flo)           | "female" | 35.0  | 1     | 0     | "113803"            | 53.1     | "C123"  | "S"      |
| 5           | 0        | 3      | "Allen, Mr. William Henry"                    | "male" | 35.0  | 0     | 0     | "373450"            | 8.05     | "Unknown" | "S"      |


In [ ]:
# print("\nДанные тестовой выборки:")
# test_df

Данные тестовой выборки:

Размер: (418, 11)

| PassengerId | Pclass | Name                                           | Sex    | Age   | SibSp | Parch | Ticket   | Fare     | Cabin | Embarked |
|-------------|--------|------------------------------------------------|--------|-------|-------|-------|----------|----------|-------|----------|
| i64         | i64    | str                                            | str    | f64   | i64   | i64   | str      | f64      | str   | str      |
| 892         | 3      | "Kelly, Mr. James"                           | "male" | 34.5  | 0     | 0     | "330911" | 7.8292   | null  | "Q"      |
| 893         | 3      | "Wilkes, Mrs. James (Ellen Needham)"        | "female" | 47.0  | 1     | 0     | "363272" | 7.0      | null  | "S"      |
| 894         | 2      | "Myles, Mr. Thomas Francis"                   | "male" | 62.0  | 0     | 0     | "240276" | 9.6875   | null  | "Q"      |
| 895         | 3      | "Wirz, Mr. Albert"                           | "male" | 27.0  | 0     | 0     | "315154" | 8.6625   | null  | "S"      |
| 896         | 3      | "Hirvonen, Mrs. Alexander (Helga)"           | "female" | 22.0  | 1     | 1     | "3101298"| 12.2875  | null  | "S"      |


### Определение типов переменных


#### Бинарные переменные

In [ ]:
# Определение бинарных переменных и вывод их уникальных значений

# Заголовок
print("Бинарные переменные и их уникальные значения:")

# Перебор всех столбцов в обучающей выборке
for column in train_df.columns:
    # Получение уникальных значений, исключая пропуски
    unique_values = train_df[column].drop_nulls().unique().to_list()

    # Проверка, что уникальных значений ровно два
    if len(unique_values) == 2:
        # Сортировка значений по количеству символов
        unique_values_sorted = sorted(unique_values, key=lambda x: len(str(x)))
        # Вывод результата
        print(f"{column}: {unique_values_sorted}")

Бинарные переменные и их уникальные значения:
Survived: [0, 1]
Sex: ['male', 'female']


In [ ]:
# Заголовок
print("Уникальные значения бинарных переменных:")

# Перебор столбцов и поиск бинарных переменных
for column in train_df.columns:
    unique_values = train_df[column].drop_nulls().unique().to_list()  # Уникальные значения без пропусков
    if len(unique_values) == 2:  # Проверка, что значений ровно два
        unique_values_sorted = sorted(unique_values, key=lambda x: len(str(x)))  # Сортировка по длине значений
        print(f"{column}: {unique_values_sorted}")

Уникальные значения бинарных переменных:
Survived: [0, 1]
Sex: ['male', 'female']


In [ ]:
# Определение бинарных переменных и вывод их уникальных значений
binary_columns = [col for col in train_df.columns if train_df[col].n_unique() == 2]

print("Уникальные значения бинарных переменных:")
for col in binary_columns:
    unique_values = sorted(train_df[col].unique().to_list(), key=lambda x: len(str(x)))
    print(f"{col}: {unique_values}")

Уникальные значения бинарных переменных:
Survived: [0, 1]
Sex: ['male', 'female']


In [ ]:
# Вывод уникальных значений для переменных с ровно двумя уникальными значениями
print("Уникальные значения бинарных переменных:")
for col in train_df.columns:
    unique_values = train_df[col].unique().drop_nulls()
    if len(unique_values) == 2:
        # Сортировка по количеству символов в значениях
        sorted_values = sorted(unique_values.to_list(), key=lambda x: len(str(x)))
        print(f"{col}: {sorted_values}")

Уникальные значения бинарных переменных:
Survived: [0, 1]
Sex: ['male', 'female']


In [ ]:
# Определение и вывод уникальных значений для бинарных переменных
print("Уникальные значения бинарных переменных:")
for col in train_df.columns:
    unique_vals = train_df[col].unique().drop_nulls()
    if len(unique_vals) == 2:
        print(f"{col}: {unique_vals.to_list()}")

Уникальные значения бинарных переменных:
Survived: [0, 1]
Sex: ['female', 'male']


In [ ]:
# Определение бинарных переменных и вывод их уникальных значений
binary_columns = [col for col in train_df.columns if train_df[col].n_unique() == 2]

for col in binary_columns:
    unique_values = train_df[col].unique().to_list()
    print(f"{col}: {unique_values}")

Survived: [0, 1]
Sex: ['male', 'female']


### Описание переменных

#### Бинарные переменные:
1. **Survived**: Целевая переменная, указывающая, выжил ли пассажир (1) или нет (0).
2. **Sex**: Пол пассажира (мужчина или женщина).

#### Числовые переменные:
3. **PassengerId**: Уникальный идентификатор пассажира.
4. **Pclass**: Класс билета (1 = Первый, 2 = Второй, 3 = Третий).
5. **Age**: Возраст пассажира.
6. **SibSp**: Количество братьев, сестер или супругов на борту.
7. **Parch**: Количество родителей или детей на борту.
8. **Fare**: Стоимость билета.

#### Категориальные переменные:
9. **Name**: Имя пассажира.
10. **Ticket**: Номер билета.
11. **Cabin**: Номер каюты.
12. **Embarked**: Порт посадки (C = Cherbourg, Q = Queenstown, S = Southampton).

### Пропуски, сцециальные символы и первый взгляд на признаки

In [ ]:
# # Проверка пропущенных значений (NaN) в обучающей выборке
# print("Проверка пропущенных значений (NaN) в обучающей выборке:")

# # Числовые столбцы с более чем 2 уникальными значениями
# numeric_columns = [col for col in train_df.columns if train_df[col].dtype in [pl.Float64, pl.Int64] and train_df[col].n_unique() > 2]
# print("\nЧисловые столбцы (непрерывные):")
# for col in numeric_columns:
#     print(f"Столбец '{col}': {train_df[col].null_count()} пропущенных значений (NaN)")

# # Категориальные столбцы с более чем 2 уникальными значениями
# categorical_columns = [col for col in train_df.columns if train_df[col].dtype == pl.Utf8 and train_df[col].n_unique() > 2]
# print("\nКатегориальные столбцы:")
# for col in categorical_columns:
#     print(f"Столбец '{col}': {train_df[col].null_count()} пропущенных значений (NaN)")

# # Бинарные столбцы
# binary_columns = [col for col in train_df.columns if train_df[col].n_unique() == 2]
# print("\nБинарные столбцы:")
# for col in binary_columns:
#     print(f"Столбец '{col}': {train_df[col].null_count()} пропущенных значений (NaN)")

# # Удаление NaN из категориальных и бинарных столбцов
# train_df_cleaned = train_df.drop_nulls(subset=categorical_columns + binary_columns)

# # Уникальные значения для категориальных и бинарных столбцов (после удаления NaN)
# print("\nУникальные значения для категориальных и бинарных столбцов (после удаления NaN):")
# for col in categorical_columns + binary_columns:
#     unique_values_sorted = sorted(train_df_cleaned[col].unique())
#     if len(unique_values_sorted) > 5:
#         print(f"Уникальные значения в столбце '{col}' (первые 5 из {len(unique_values_sorted)}): {unique_values_sorted[:5]}")
#     else:
#         print(f"Уникальные значения в столбце '{col}': {unique_values_sorted}")

**Проверка пропущенных значений (NaN) в обучающей выборке:**

**Числовые столбцы (непрерывные):**
- Столбец 'PassengerId': 0 пропущенных значений (NaN)
- Столбец 'Pclass': 0 пропущенных значений (NaN)
- Столбец 'Age': 177 пропущенных значений (NaN)
- Столбец 'SibSp': 0 пропущенных значений (NaN)
- Столбец 'Parch': 0 пропущенных значений (NaN)
- Столбец 'Fare': 0 пропущенных значений (NaN)

**Категориальные столбцы:**
- Столбец 'Name': 0 пропущенных значений (NaN)
- Столбец 'Ticket': 0 пропущенных значений (NaN)
- Столбец 'Cabin': 687 пропущенных значений (NaN)
- Столбец 'Embarked': 2 пропущенных значения (NaN)

**Бинарные столбцы:**
- Столбец 'Survived': 0 пропущенных значений (NaN)
- Столбец 'Sex': 0 пропущенных значений (NaN)

**Уникальные значения для категориальных и бинарных столбцов (после удаления NaN):**
- Уникальные значения в столбце 'Name' (первые 5 из 202): ['Allen, Miss. Elisabeth Walton', 'Allison, Master. Hudson Trevor', 'Allison, Miss. Helen Loraine', 'Allison, Mrs. Hudson J C (Bessie Waldo Daniels)', 'Anderson, Mr. Harry']
- Уникальные значения в столбце 'Ticket' (первые 5 из 141): ['110152', '110413', '110465', '110564', '110813']
- Уникальные значения в столбце 'Cabin' (первые 5 из 146): ['A10', 'A14', 'A16', 'A19', 'A20']
- Уникальные значения в столбце 'Embarked': ['C', 'Q', 'S']
- Уникальные значения в столбце 'Survived': [0, 1]
- Уникальные значения в столбце 'Sex': ['female', 'male']

In [ ]:
# Проверка пропущенных значений (NaN) в обучающей выборке
print("Проверка пропущенных значений (NaN) в обучающей выборке:")

# Числовые столбцы (непрерывные)
numeric_cols = train_df.select(pl.col(pl.NUMERIC_DTYPES)).columns
numeric_cols = [col for col in numeric_cols if train_df[col].n_unique() > 2 and col != "PassengerId"]
print("\nЧисловые столбцы (>2 уникальных значений):")
print(train_df.select(numeric_cols).null_count())

# Категориальные столбцы (номинальные + ординальные)
categorical_cols = [
    col for col in train_df.columns
    if (train_df[col].n_unique() <= 10) and (col not in numeric_cols + ["Sex", "Survived", "PassengerId"])
]
print("\nКатегориальные столбцы:")
print(train_df.select(categorical_cols).null_count())

# Бинарные столбцы
binary_cols = [col for col in train_df.columns if train_df[col].n_unique() == 2]
print("\nБинарные столбцы:")
print(train_df.select(binary_cols).null_count())

# Анализ уникальных значений (категории + специальные символы)
print("\nАнализ уникальных значений:")

for column in categorical_cols + binary_cols:
    unique_values = train_df[column].unique().sort().to_list()
    print(f"\nСтолбец: {column}")
    print(f"Тип данных: {train_df.schema[column]}")

    if len(unique_values) > 5:
        print(f"Уникальные значения (первые 5 из {len(unique_values)}): {unique_values[:5]}")
    else:
        print(f"Уникальные значения: {unique_values}")

    # Проверка на специальные символы
    special_chars = [val for val in unique_values if isinstance(val, str) and not val.strip().isalnum()]
    if special_chars:
        print(f"⚠️ Обнаружены специальные символы: {special_chars[:3]}")

Проверка пропущенных значений (NaN) в обучающей выборке:

Числовые столбцы (>2 уникальных значений):
shape: (1, 5)
┌────────┬─────┬───────┬───────┬──────┐
│ Pclass ┆ Age ┆ SibSp ┆ Parch ┆ Fare │
│ ---    ┆ --- ┆ ---   ┆ ---   ┆ ---  │
│ u32    ┆ u32 ┆ u32   ┆ u32   ┆ u32  │
╞════════╪═════╪═══════╪═══════╪══════╡
│ 0      ┆ 177 ┆ 0     ┆ 0     ┆ 0    │
└────────┴─────┴───────┴───────┴──────┘

Категориальные столбцы:
shape: (1, 1)
┌──────────┐
│ Embarked │
│ ---      │
│ u32      │
╞══════════╡
│ 2        │
└──────────┘

Бинарные столбцы:
shape: (1, 2)
┌──────────┬─────┐
│ Survived ┆ Sex │
│ ---      ┆ --- │
│ u32      ┆ u32 │
╞══════════╪═════╡
│ 0        ┆ 0   │
└──────────┴─────┘

Анализ уникальных значений:

Столбец: Embarked
Тип данных: String
Уникальные значения: [None, 'C', 'Q', 'S']

Столбец: Survived
Тип данных: Int64
Уникальные значения: [0, 1]

Столбец: Sex
Тип данных: String
Уникальные значения: ['female', 'male']


<ipython-input-51-eed8bbadaeb1>:5: DeprecationWarning: `NUMERIC_DTYPES` is deprecated. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  numeric_cols = train_df.select(pl.col(pl.NUMERIC_DTYPES)).columns


### Ключевая информация о переменных



#### Особенности и интересные наблюдения:
1. **Возраст (Age)**: Значительное количество пропущенных значений (177), что может потребовать заполнения или удаления строк.  
2. **Каюта (Cabin)**: Большое количество пропусков (687), что делает этот столбец малопригодным для анализа без дополнительной обработки.  
3. **Порт посадки (Embarked)**: Небольшое количество пропусков (2), которые можно легко заполнить.  
4. **Билет (Ticket)**: Номера билетов содержат как числовые, так и текстовые данные, что может потребовать дополнительной обработки.  
5. **Класс билета (Pclass)**: Распределение по классам может быть важным фактором для анализа выживаемости.  

## 3. Подготовка данных (Data Preparation)

### Очистка данных

In [ ]:
# Очистка данных

# 1. Заполнение пропусков в Age медианным значением
median_age = train_df['Age'].median()  # Вычисляем медиану возраста
train_df = train_df.with_columns(pl.col('Age').fill_null(median_age))  # Заполняем пропуски

# 2. Заполнение пропусков в Cabin значением "Unknown"
train_df = train_df.with_columns(pl.col('Cabin').fill_null("Unknown"))  # Заполняем пропуски

# 3. Заполнение пропусков в Embarked наиболее частым значением (модой)
mode_embarked = train_df['Embarked'].mode()[0]  # Вычисляем моду (наиболее частое значение)
train_df = train_df.with_columns(pl.col('Embarked').fill_null(mode_embarked))  # Заполняем пропуски

In [ ]:
# # Проверка результата
# print("Проверка пропусков после очистки:")
# print(train_df.null_count())

Проверка пропусков после очистки:

Размер: (1, 12)

| PassengerId | Survived | Pclass | Name | Sex | Age | SibSp | Parch | Ticket | Fare | Cabin | Embarked |
|-------------|----------|--------|------|-----|-----|-------|-------|--------|------|-------|----------|
| u32         | u32      | u32    | u32  | u32 | u32 | u32   | u32   | u32    | u32  | u32   | u32      |
| 0           | 0        | 0      | 0    | 0   | 0   | 0     | 0     | 0      | 0    | 0     | 0        |
